# MYSTRAN and NASTRAN airplane demo 

Read in mystran and nastran finite element models and results and compare to the built-in airplane beam model 


# Imports

In [ ]:
import sdynpy as sdpy

In [ ]:
import pathlib
import pandas as pd
import numpy as np
import importlib
import sys, pathlib
import matplotlib.pyplot as plt

In [ ]:
curdir = pathlib.Path().cwd()
curdir

In [ ]:
print(sdpy.__file__)   # should be ...\github\sdynpy\src\sdynpy\__init__.py

In [ ]:
repo_root = curdir.parents[2]
repo_root

In [ ]:
# reload sdynpy module if changing the source code

In [ ]:
%load_ext autoreload
%autoreload 2

# General Plotting settings

In [ ]:
# plotting params
plot_kwargs={'node_size':1, 'view_from':[-1,1,1]}

# sdynpy demo airplane

In [ ]:
import sdynpy.demo as demo
geometry_sdy = demo.beam_airplane.geometry
system_sdy = demo.beam_airplane.system

# Get the airplane system and geometry from the demo subpackage
geometry_sdy = demo.beam_airplane.geometry

In [ ]:
shapes_sdy = system_sdy.eigensolution(maximum_frequency=10)
shapes_sdy = shapes_sdy[6:]
shapes_sdy.damping = 0.02
shapes_sdy

In [ ]:
shapes_sdy.save('demo_airplane_geometry_sdy_shapes.npy')

In [ ]:
# Create a modal model to make integration faster
modal_system_sdy = shapes_sdy[6:].system()

In [ ]:
geometry_sdy.node[:10]

In [ ]:
geometry_sdy.plot(label_nodes=True)

## export demo model to csv to read externally

In [ ]:
n = geometry_sdy.node
df = pd.DataFrame({
                    'id': n.id,
                    'x': n.coordinate[:, 0],
                    'y': n.coordinate[:, 1],
                    'z': n.coordinate[:, 2],
                    'def_cs': n.def_cs,   # CP: frame the position is defined in
                    'disp_cs': n.disp_cs, # CD: frame the DOFs/displacements are defined in
                    'color': n.color,
                    })
df.set_index('id')
df.to_csv('demo_airplane_grids_export.csv')

# Mystran FE Model

## Read Mystran dat and f06 demo airplane model

In [ ]:
mystrandatfilename = repo_root.joinpath('docs/book/example_models/sdynpy_airplane_modes/sdynpy_airplane_modes_mystran.dat')
print(mystrandatfilename)
geometry_mys, shapes_mys = sdpy.mystran.read_mystran(mystrandatfilename, source='f06', damping=0.005)
# print(shapes.frequency)
shapes_mys = shapes_mys[6:]
shapes_mys.damping = 0.02
shapes_mys

## Alternatively, read msytran ou4 file

In [ ]:
# The OU4 results should match the F06 results
_, shapes_ou4 = sdpy.mystran.read_mystran(mystrandatfilename, source='ou4', damping=0.005)
shapes_ou4 = shapes_ou4[6:]
shapes_ou4.damping = 0.02
shapes_ou4

## save sdynpy geometry for use later

In [ ]:
geometry_mys.save('demo_airplane_geometry.npy')
geometry_mys.write_to_unv('demo_airplane_geometry.unv')

In [ ]:
sdpy.fileio.sdynpy_uff.readunv('demo_airplane_geometry.unv')

## Save MYSTRAN shapes to sdynpy file

In [ ]:
shapes_mys.save('demo_airplane_geometry_mystran_shapes.npy')
shapes_mys = sdpy.shape.load('demo_airplane_geometry_mystran_shapes.npy')

## plot mystran model with grids

In [ ]:
geometry_mys.plot(label_nodes=True, node_size=8, label_font_size=10)

In [ ]:
# Every translation DOF at every node (default)
geometry_mys.plot_coordinate(label_dofs=True, label_font_size=10, arrow_scale=0.02)

In [ ]:
# only plot Z+
coords = sdpy.coordinate_array(geometry_mys.node.id, 'Z+')
geometry_mys.plot_coordinate(coords, label_dofs=True, arrow_scale=0.02)

In [ ]:
# Specific DOFs, e.g. all three translations and rotations at selected nodes
# excitation_dofs = sdpy.coordinate_array([241, 201], ['Z+', 'Z+', ])
# excitation_dofs = sdpy.coordinate_array( string_array = ['241Z+','201Z+'])
excitation_dofs = sdpy.coordinate_array([241, 201], 'Z+')
geometry_mys.plot_coordinate(excitation_dofs, label_dofs=True, arrow_scale=0.03, label_font_size=10, plot_kwargs=plot_kwargs)

In [ ]:
geometry_mys.node.disp_cs

In [ ]:
# Only nodes that use csys 21, which is the right vstab
nodes = geometry_mys.node.id[geometry_mys.node.disp_cs == 21]
print(nodes)
geometry_mys.plot_coordinate(sdpy.coordinate.from_nodelist(nodes), label_dofs=True, arrow_scale=0.06)

## plot response points

In [ ]:
# response_dofs = sdpy.coordinate.from_nodelist(geometry_mys.node.id)
response_dofs = sdpy.coordinate_array( string_array = ['275Z+','249Z+','101Z+','101Y+'])
response_dofs

In [ ]:
geometry_mys.plot_coordinate(response_dofs,label_dofs=True, arrow_scale=0.03, label_font_size=10, 
                         plot_kwargs=plot_kwargs)

## animate mystran f06 modes

In [ ]:
p = geometry_mys.plot_shape(shapes_mys, plot_kwargs=plot_kwargs, 
                        background_plotter_kwargs={'editor': False, 'window_size': (800, 600)})

comment = p.text                      # SDynPy's mode/frequency/damping label
# Add your own text (any corner or position)
p.add_text('sdynpy demo airplane mode shapes', position='upper_right', font_size=14, color='k', name='title')
# Or place it in pixel/viewport coordinates
p.add_text('MYSTRAN solver', position='lower_right', font_size=12, name='tag')  # position=(10, 10)
# p.app_window.resize(1600, 900)
p.add_axes()                          # small XYZ triad in the corner
p.text = comment                      # restore it so mode changes update the right actor

## export modeshapes to gif

In [ ]:
#### export single gif of animation
p = geometry_mys.plot_shape(shapes_mys[0], plot_kwargs=plot_kwargs, 
                        background_plotter_kwargs={'editor': False, 'window_size': (800, 600)})
p.save_animation('mode_1.gif', frames=30, frame_rate=20)            # current shape

In [ ]:
### export all mode gifs unattended
p = geometry_mys.plot_shape(shapes_mys[:5], plot_kwargs=plot_kwargs, 
                        background_plotter_kwargs={'window_size': (800, 600),'editor': False, 'show': False})
p.save_animation_all_shapes('mode_{:}.gif')
p.close()

## Convert MSYTRAN model shapes to a modal system   

In [ ]:
modal_system_mys = shapes_mys.system() 
modal_system_mys

In [ ]:
# Plot the modal system matrices
fig,ax = plt.subplots(1,4,num='Modal System Matrices',figsize=(12,3))
# Transformation
timg = ax[0].imshow(modal_system_mys.transformation)
ax[0].set_title('Transformation')
ax[0].set_ylabel('Physical DoF')
ax[0].set_xlabel('Modal DoF')
plt.colorbar(timg,ax=ax[0])
# Mass
mimg = ax[1].imshow(modal_system_mys.mass)
ax[1].set_title('Mass')
ax[1].set_ylabel('Modal DoF')
ax[1].set_xlabel('Modal DoF')
plt.colorbar(mimg,ax=ax[1])
# Damping
dimg = ax[2].imshow(modal_system_mys.damping)
ax[2].set_title('Damping')
ax[2].set_ylabel('Modal DoF')
ax[2].set_xlabel('Modal DoF')
plt.colorbar(dimg,ax=ax[2])
# Stiffness
simg = ax[3].imshow(modal_system_mys.stiffness)
ax[3].set_title('Stiffness')
ax[3].set_ylabel('Modal DoF')
ax[3].set_xlabel('Modal DoF')
plt.colorbar(simg,ax=ax[3])
fig.tight_layout()

# NASTRAN FE model and results from f06

In [ ]:
nastrandatfilename = repo_root.joinpath('docs/book/example_models/sdynpy_airplane_modes/sdynpy_airplane_modes_nastran.dat')
print(nastrandatfilename)
geometry_nas, shapes_nas = sdpy.nastran.read_nastran(nastrandatfilename, source='f06', damping=0.005)
# print(shapes.frequency)
shapes_nas = shapes_nas[6:]
shapes_nas.damping = 0.02
shapes_nas

## alternately, read nastran pch

In [ ]:
geometry_nas, shapes_nas = sdpy.nastran.read_nastran(nastrandatfilename, source='pch', damping=0.005)
shapes_nas = shapes_nas[6:]
shapes_nas.damping = 0.02
shapes_nas

In [ ]:
geometry_nas.plot_shape(shapes_nas)

## compare mystran and nastran models

In [ ]:
# mystran vs nastran
sdpy.shape.mac(shapes_mys, shapes_nas).diagonal().round(2)

In [ ]:
print('NASTRAN to MYSTRAN Results Comparison')
print(sdpy.shape.shape_comparison_table(
        shapes_mys,
        shapes_nas,
        percent_error_format='{:0.3f}%'))

# Mode Shapes and Frequencies Compare  

the mystran and nastran models are nearly identical, where are the sdynpy airplane beam model is off a bit

In [ ]:
shapes_sdy = sdpy.shape.load('demo_airplane_geometry_mystran_shapes.npy')
shapes_mys = sdpy.shape.load('demo_airplane_geometry_sdy_shapes.npy')

In [ ]:
shapes_sdy = shapes_sdy[:9]
shapes_sdy

In [ ]:
shapes_mys

In [ ]:
# Compute MACs
mac_mys_sdy = sdpy.shape.mac(shapes_mys,shapes_sdy)

In [ ]:
# mystran vs sdynpy
mac_mys_sdy.diagonal().round(2)

In [ ]:
# Create a figure
fig,ax = plt.subplots(1,1,num='MAC Matrices', figsize=(6,6))
# Plot the PolyPy MAC
sdpy.matrix_plot(mac_mys_sdy,ax=ax)
ax.set_xlabel('mystran shapes')
ax.set_ylabel('sdynpy shapes')
ax.set_title('Airplane beam demo MYSTRAN vs SDYNPY mode shapes compare')

In [ ]:
print('Sdnpy to mystran Results Comparison')
print(sdpy.shape.shape_comparison_table(
        shapes_mys[:8],
        shapes_sdy[:8],
        percent_error_format='{:0.3f}%'))

## animate mode shapes together

In [ ]:
# Overlay shapes
overlaid_geometry,overlaid_shapes = sdpy.shape.overlay_shapes(
    (geometry_mys,geometry_sdy),
    (shapes_mys,shapes_sdy),
    color_override=[1,7])
# Plot the overlaid shapes on the overlaid geometry
overlaid_geometry.plot_shape(overlaid_shapes)

## mystran compute FRF from shapes

In [ ]:
# Acceleration FRFs between any pair of degrees of freedom from the unconstrained DOFs.

frequencies = np.arange(0, 10, 0.01)
responses = sdpy.coordinate_array([101, 201, 241], 'Z+')
# responses = sdpy.coordinate.from_nodelist(geometry_mys.node.id) # all frfs
references = sdpy.coordinate_array(201, 'Z+')
frfs_airplane_mystran = shapes_mys.compute_frf(frequencies, responses, references)
frfs_airplane_mystran.plot();

## mystran compute frfs from system

In [ ]:
responses = sdpy.coordinate_array([101, 201, 241], 'Z+')
references = sdpy.coordinate_array([201], 'Z+')
# From the reduced system with damping added
frfs_airplane_mystran = modal_system_mys.frequency_response(
                                                    frequencies,
                                                    responses,
                                                    references,
                                                    displacement_derivative=2)

frfs_airplane_mystran.plot()

In [ ]:
frfs_airplane_mystran

## manual plot of FRFs

In [ ]:
frf_drive = frfs_airplane_mystran.get_drive_points()
# frf = frfs_modal_system[0]
fig,ax = plt.subplots(2,1,figsize=(10,6))

frf_ang = np.angle(frf_drive.ordinate)*180/np.pi
frf_mag = np.abs(frf_drive.ordinate)

ax[0].plot(frf_drive.abscissa[0], frf_ang[0])
ax[1].plot(frf_drive.abscissa[0], frf_mag[0])
ax[1].set_yscale('log')

ax[0].set_ylabel('Phase (deg)')
ax[1].set_ylabel('Magnitude (m/s$^2$/N)')
ax[0].legend(['Modal System']);
ax[1].legend(['Modal System']);
ax[0].set_title(f"Drive Point {frf_drive.response_coordinate.string_array()}")
ax[1].set_xlabel('Frequency (Hz)')

## compare FRF plots

In [ ]:
frfs_airplane_sdynpy = shapes_sdy.compute_frf(frequencies, responses, references)

In [ ]:
frfs_airplane_sdynpy

In [ ]:
frfs_airplane_mystran

In [ ]:
## interactive plotting 
# frfs_airplane_mystran.gui_plot()
# frfs_airplane_sdynpy.gui_plot()

In [ ]:
## static plots
# frfs_airplane_mystran[0,0].plot()
# frfs_airplane_sdynpy[0,0].plot()

In [ ]:
frequencies = np.arange(0, 10, 0.01)

responses = sdpy.coordinate_array([101, 201, 241], 'Z+')
references = sdpy.coordinate_array([201], 'Z+')

frfs_airplane_mystran = shapes_mys.compute_frf(frequencies, responses, references)
frfs_airplane_sdynpy = shapes_sdy.compute_frf(frequencies, responses, references)

for n in range(frfs_airplane_mystran.shape[0]):
    # n = 1
    
    res = frfs_airplane_mystran[n,0].response_coordinate.string_array()
    ref = frfs_airplane_mystran[n,0].reference_coordinate.string_array()
    frf_name_mys = f'frf_{ref}_{res}_mystran'
    
    res = frfs_airplane_sdynpy[n,0].response_coordinate.string_array()
    ref = frfs_airplane_sdynpy[n,0].reference_coordinate.string_array()
    frf_name_sdy = f'frf_{ref}_{res}_sdynpy'
    
    fig1,ax1 = plt.subplots(1,1,figsize=(10,4))
    ax1.plot(frfs_airplane_mystran.abscissa[n,0],
             np.abs(frfs_airplane_mystran.ordinate[n,0]), 
             label=frf_name_mys)
    
    ax1.plot(frfs_airplane_sdynpy.abscissa[n,0],
             
             np.abs(frfs_airplane_sdynpy.ordinate[n,0]),
             label=frf_name_sdy)
    
    ax1.set_title('Model Compare Accel FRFs')
    ax1.set_xlabel('freq,hz')
    ax1.set_ylabel('mag,accel')
    ax1.set_yscale('log')
    ax1.set_ylim([1e-5,0.1])
    ax1.legend();

## Compare FRFS in guiplot

In [ ]:
sdpy.GUIPlot( 
                mystran_airplane = frfs_airplane_mystran,
                sdynpy_airplane = frfs_airplane_sdynpy,
            )

## plot deflection shape from frfs

In [ ]:
frequencies = np.arange(0.01, 10, 0.01)
responses = sdpy.coordinate.from_nodelist(geometry_mys.node.id) # all frfs
references = sdpy.coordinate_array(201, 'Z+')
frfs = shapes_mys.compute_frf(frequencies, responses, references)

In [ ]:
geometry_mys.plot_deflection_shape(frfs[:,0])